In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense,
    RandomRotation, RandomFlip, RandomZoom
)
from sklearn.model_selection import train_test_split

# 1. Load CIFAR-10 Dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print("Training Dataset Shape:", x_train.shape)
print("Testing Dataset Shape:", x_test.shape)

# 2. Normalize Images
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = y_train.flatten()
y_test = y_test.flatten()

# Split training and validation data
X_train, X_val, Y_train, Y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

# 3. Create CNN Model
def create_model(augmentation=False):
    model = Sequential([
        Input(shape=(32, 32, 3))
    ])

    # Data Augmentation
    if augmentation:
        model.add(RandomRotation(0.1))
        model.add(RandomFlip("horizontal"))
        model.add(RandomZoom(0.1))

    # CNN Layers
    model.add(Conv2D(32, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(Flatten())
    model.add(Dense(128, activation="relu"))
    model.add(Dense(10, activation="softmax"))

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# 4. Train Without Augmentation
print("\nTraining Without Data Augmentation")
model1 = create_model(False)

start = time.time()
history1 = model1.fit(
    X_train, Y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_val, Y_val),
    verbose=1
)
time1 = time.time() - start

loss1, accuracy1 = model1.evaluate(x_test, y_test, verbose=0)

# 5. Train With Augmentation
print("\nTraining With Data Augmentation")
model2 = create_model(True)

start = time.time()
history2 = model2.fit(
    X_train, Y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_val, Y_val),
    verbose=1
)
time2 = time.time() - start

loss2, accuracy2 = model2.evaluate(x_test, y_test, verbose=0)

# 6. Compare Both Models
results = pd.DataFrame({
    "Model": ["Without Augmentation", "With Augmentation"],
    "Training Accuracy (%)": [
        round(history1.history["accuracy"][-1] * 100, 2),
        round(history2.history["accuracy"][-1] * 100, 2)
    ],
    "Validation Accuracy (%)": [
        round(history1.history["val_accuracy"][-1] * 100, 2),
        round(history2.history["val_accuracy"][-1] * 100, 2)
    ],
    "Test Accuracy (%)": [
        round(accuracy1 * 100, 2),
        round(accuracy2 * 100, 2)
    ],
    "Training Time (seconds)": [
        round(time1, 2),
        round(time2, 2)
    ]
})

print("\nFINAL MODEL COMPARISON")
print(results.to_string(index=False))

# 7. Plot Accuracy Comparison
plt.figure(figsize=(8, 5))
plt.plot(history1.history["val_accuracy"], label="Without Augmentation")
plt.plot(history2.history["val_accuracy"], label="With Augmentation")
plt.title("CNN Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid()
plt.show()